## Building Kepler dashboard for socioeconomic data

This notebook is dedicated for testing the Kepler.gl dashboard for socioeconomic DE pipeline project.

In [48]:
import boto3
import awswrangler as wr
import os

In [57]:
import geopandas as gpd
from shapely import wkt
import numpy as np
from lonboard import Map, SolidPolygonLayer
from lonboard.colormap import apply_continuous_cmap
import matplotlib.cm as cm
from lonboard._viewport import compute_view
from lonboard.basemap import MaplibreBasemap, CartoStyle

In [50]:
os.environ['AWS_PROFILE'] = 'my-dev-profile'

In [52]:
session = boto3.Session(region_name='us-east-1')

df = wr.athena.read_sql_query(
    sql="""
        SELECT
            geography_id,
            geography_name,
            total_population,
            median_household_income,
            poverty_rate,
            bachelors_plus_rate,
            renter_rate,
            remote_work_rate,
            geometry_wkt
        FROM mart_socioeconomic_tracts
        WHERE state_fips = '06'
        AND survey_year = 2024
    """,
    database="population_demographics_gold_marts",
    workgroup="population-demographics",
    boto3_session=session
)

print(f"Loaded {len(df)} rows")
print(df.head(2))

# Convert to GeoDataFrame
df['geometry'] = df['geometry_wkt'].apply(wkt.loads)
# Round poverty_rate etc. before creating GeoDataFrame
df['poverty_rate'] = df['poverty_rate'].round(1)
df['bachelors_plus_rate'] = df['bachelors_plus_rate'].round(1)
df['renter_rate'] = df['renter_rate'].round(1)
df['remote_work_rate'] = df['remote_work_rate'].round(1)
gdf = gpd.GeoDataFrame(df.drop(columns=['geometry_wkt']), geometry='geometry', crs='EPSG:4326')

Loaded 9129 rows
  geography_id                                    geography_name  \
0  06001403702  Census Tract 4037.02; Alameda County; California   
1  06001405800     Census Tract 4058; Alameda County; California   

   total_population  median_household_income  poverty_rate  \
0              2198                   117470          6.28   
1              4590                    68378         28.36   

   bachelors_plus_rate  renter_rate  remote_work_rate  \
0                74.51        81.73             36.90   
1                20.29        56.24              7.61   

                                        geometry_wkt  
0  POLYGON ((-122.256336 37.808538, -122.256309 3...  
1  POLYGON ((-122.239646 37.795268, -122.239084 3...  


Let's try to render some Polygons

In [59]:
values = gdf['median_household_income'].fillna(0).values.astype(float)

# Use percentile clipping to handle outliers skewing the colormap
p5 = np.percentile(values, 5)
p95 = np.percentile(values, 95)
clipped = np.clip(values, p5, p95)
normalized = (clipped - clipped.min()) / (clipped.max() - clipped.min())

colors = apply_continuous_cmap(normalized, cm.YlOrRd, alpha=170)

layer = SolidPolygonLayer.from_geopandas(
    gdf,
    get_fill_color=colors,
    pickable=True,
)

m = Map(
    layer,
    basemap=MaplibreBasemap(
        style=CartoStyle.Positron,
        mode="reverse-controlled"  # basemap renders on top, clips water naturally
    ),
)
m.to_html("ca_socioeconomic_map.html")

In [25]:
print(gdf.shape)
print(gdf.geometry.is_valid.all())
print(gdf.geometry.is_empty.any())
print(gdf.crs)

(9129, 10)
True
False
EPSG:4326


In [60]:
# Generate GeoJSON from your GeoDataFrame
import json

# Add color to each feature
def normalize(val, min_val, max_val):
    return (val - min_val) / (max_val - min_val) if max_val > min_val else 0

import matplotlib.cm as cm
import matplotlib.colors as mcolors

p5 = float(np.percentile(values, 5))
p95 = float(np.percentile(values, 95))

features = []
for _, row in gdf.iterrows():
    val = float(row['median_household_income']) if row['median_household_income'] else 0
    clipped_val = max(p5, min(p95, val))
    norm = (clipped_val - p5) / (p95 - p5)
    rgba = cm.YlOrRd(norm)
    color = [int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255), 170]
    
    features.append({
        "type": "Feature",
        "geometry": row['geometry'].__geo_interface__,
        "properties": {
            "geography_id": row['geography_id'],
            "geography_name": row['geography_name'],
            "total_population": row['total_population'],
            "median_household_income": row['median_household_income'],
            "poverty_rate": row['poverty_rate'],
            "bachelors_plus_rate": row['bachelors_plus_rate'],
            "renter_rate": row['renter_rate'],
            "remote_work_rate": row['remote_work_rate'],
            "fill_color": color
        }
    })

geojson = {"type": "FeatureCollection", "features": features}
geojson_str = json.dumps(geojson)

html = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>CA Socioeconomic Dashboard</title>
  <meta name="viewport" content="initial-scale=1,maximum-scale=1,user-scalable=no">
  <script src="https://unpkg.com/deck.gl@latest/dist.min.js"></script>
  <script src="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.js"></script>
  <link href="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.css" rel="stylesheet">
  <style>
    body {{ margin: 0; padding: 0; }}
    #map {{ width: 100vw; height: 100vh; }}
    #tooltip {{
      position: absolute;
      background: rgba(0,0,0,0.8);
      color: white;
      padding: 10px;
      border-radius: 4px;
      font-family: monospace;
      font-size: 12px;
      pointer-events: none;
      display: none;
      max-width: 250px;
    }}
    #legend {{
      position: absolute;
      bottom: 30px;
      left: 20px;
      background: rgba(255,255,255,0.9);
      padding: 10px;
      border-radius: 4px;
      font-family: sans-serif;
      font-size: 12px;
    }}
  </style>
</head>
<body>
  <div id="map"></div>
  <div id="tooltip"></div>
  <div id="legend">
    <b>Median Household Income</b><br>
    <div style="background:linear-gradient(to right,#ffffb2,#fd8d3c,#bd0026);height:12px;width:150px;margin:4px 0"></div>
    <div style="display:flex;justify-content:space-between;width:150px">
      <span>${int(p5):,}</span><span>${int(p95):,}+</span>
    </div>
  </div>

  <script>
    const geojsonData = {geojson_str};

    const map = new maplibregl.Map({{
      container: 'map',
      style: 'https://basemaps.cartocdn.com/gl/positron-gl-style/style.json',
      center: [-119.4179, 36.7783],
      zoom: 6
    }});

    const tooltip = document.getElementById('tooltip');

    const deckOverlay = new deck.MapboxOverlay({{
      interleaved: true,
      layers: [
        new deck.GeoJsonLayer({{
          id: 'tracts',
          data: geojsonData,
          filled: true,
          stroked: true,
          getFillColor: f => f.properties.fill_color,
          getLineColor: [0, 0, 0, 30],
          getLineWidth: 20,
          lineWidthMinPixels: 0.5,
          pickable: true,
          beforeId: 'watername_ocean',
          onHover: ({{object, x, y}}) => {{
            if (object) {{
              const p = object.properties;
              tooltip.style.display = 'block';
              tooltip.style.left = x + 10 + 'px';
              tooltip.style.top = y + 10 + 'px';
              tooltip.innerHTML = `
                <b>${{p.geography_name}}</b><br>
                Population: ${{p.total_population?.toLocaleString()}}<br>
                Median Income: $${{p.median_household_income?.toLocaleString()}}<br>
                Poverty Rate: ${{p.poverty_rate}}%<br>
                Bachelors+: ${{p.bachelors_plus_rate}}%<br>
                Renter Rate: ${{p.renter_rate}}%<br>
                Remote Work: ${{p.remote_work_rate}}%
              `;
            }} else {{
              tooltip.style.display = 'none';
            }}
          }}
        }})
      ]
    }});

    map.addControl(deckOverlay);
    map.addControl(new maplibregl.NavigationControl());
  </script>
</body>
</html>"""

with open("ca_socioeconomic_dashboard.html", "w") as f:
    f.write(html)

print("Dashboard saved to ca_socioeconomic_dashboard.html")

Dashboard saved to ca_socioeconomic_dashboard.html


Let's try loading all three geography levels from Athena to verify it works.

In [61]:
# Test all three queries
import awswrangler as wr
import boto3
import os

os.environ['AWS_PROFILE'] = 'my-dev-profile'
session = boto3.Session(region_name='us-east-1')

DB = "population_demographics_gold_marts"
WG = "population-demographics"

# National - states
df_states = wr.athena.read_sql_query(
    sql="""
        SELECT geography_id, geography_name, state_fips,
               total_population, median_household_income,
               poverty_rate, bachelors_plus_rate,
               renter_rate, remote_work_rate, geometry_wkt
        FROM mart_socioeconomic_states
        WHERE survey_year = 2024
    """,
    database=DB, workgroup=WG, boto3_session=session
)

# State detail - counties (CA)
df_counties = wr.athena.read_sql_query(
    sql="""
        SELECT geography_id, geography_name, state_fips, county_fips,
               total_population, median_household_income,
               poverty_rate, bachelors_plus_rate,
               renter_rate, remote_work_rate, geometry_wkt
        FROM mart_socioeconomic_counties
        WHERE state_fips = '06' AND survey_year = 2024
    """,
    database=DB, workgroup=WG, boto3_session=session
)

print(f"States: {len(df_states)} rows")
print(f"Counties CA: {len(df_counties)} rows")
print(f"States geometry sample: {df_states['geometry_wkt'].iloc[0][:50]}")
print(f"Counties geometry sample: {df_counties['geometry_wkt'].iloc[0][:50]}")

States: 52 rows
Counties CA: 58 rows
States geometry sample: POLYGON ((-85.127329 31.762563, -85.12753 31.76216
Counties geometry sample: POLYGON ((-121.439911 38.255531, -121.440023 38.25


In [3]:
import json
import os
import numpy as np
import matplotlib.cm as cm
import awswrangler as wr
import boto3
import pandas as pd
from shapely import wkt as shapely_wkt

# --- Config ---
os.environ['AWS_PROFILE'] = 'my-dev-profile'
session = boto3.Session(region_name='us-east-1')
DB = "population_demographics_gold_marts"
WG = "population-demographics"
OUTPUT_DIR = "../dashboard/static/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

YEARS = list(range(2012, 2025))

METRICS = {
    "median_household_income": {
        "label": "Median Household Income",
        "description": "Middle income among all households. Half earn more, half earn less.",
        "format": "currency",
        "colormap": cm.YlOrRd,
    },
    "poverty_rate": {
        "label": "Poverty Rate",
        "description": "% of population living below the federal poverty line.",
        "format": "percent",
        "colormap": cm.YlOrRd,
    },
    "bachelors_plus_rate": {
        "label": "Bachelor's+ Rate",
        "description": "% of adults 25+ with a bachelor's degree or higher.",
        "format": "percent",
        "colormap": cm.YlGnBu,
    },
    "renter_rate": {
        "label": "Renter Rate",
        "description": "% of occupied housing units that are renter-occupied.",
        "format": "percent",
        "colormap": cm.PuRd,
    },
    "remote_work_rate": {
        "label": "Remote Work Rate",
        "description": "% of workers who worked from home in the past week.",
        "format": "percent",
        "colormap": cm.BuPu,
    },
}

# --- Helper functions ---

def compute_colors(values, cmap):
    p5 = float(np.percentile(values, 5))
    p95 = float(np.percentile(values, 95))
    clipped = np.clip(values, p5, p95)
    normalized = (clipped - p5) / (p95 - p5) if p95 > p5 else np.zeros_like(clipped)
    colors = []
    for n in normalized:
        rgba = cmap(float(n))
        colors.append([int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255), 180])
    return colors, p5, p95


def df_to_geojson(df, metric):
    cmap = METRICS[metric]["colormap"]
    values = df[metric].fillna(0).values.astype(float)
    colors, p5, p95 = compute_colors(values, cmap)
    geometries = [shapely_wkt.loads(g).__geo_interface__ for g in df['geometry_wkt']]

    cols = [c for c in df.columns if c != 'geometry_wkt']
    features = []
    for i, (_, row) in enumerate(df[cols].iterrows()):
        props = {}
        for k, v in row.items():
            if not isinstance(v, str) and pd.isna(v):
                props[k] = None
            elif hasattr(v, 'item'):
                props[k] = v.item()
            else:
                props[k] = v
        props['fill_color'] = colors[i]
        features.append({
            "type": "Feature",
            "geometry": geometries[i],
            "properties": props
        })

    return {
        "type": "FeatureCollection",
        "features": features,
        "meta": {
            "min": p5,
            "max": p95,
            "metric": metric,
            "label": METRICS[metric]["label"],
            "format": METRICS[metric]["format"],
            "description": METRICS[metric]["description"],
        }
    }


def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data, f)

Generating states json files

In [4]:
import gc

YEARS = list(range(2012, 2025))

print("=== Regenerating states data ===")
for year in YEARS:
    try:
        df = wr.athena.read_sql_query(
            sql=f"""
                SELECT geography_id, geography_name, state_fips,
                       survey_year, total_population, median_age,
                       median_household_income, median_home_value,
                       median_gross_rent, pct_white_alone, pct_black_alone,
                       pct_asian_alone, pct_hispanic_or_latino,
                       poverty_rate, bachelors_plus_rate, high_school_rate,
                       owner_occupancy_rate, renter_rate, drove_alone_rate,
                       walked_to_work_rate, remote_work_rate,
                       centroid_lat, centroid_lon,
                       geometry_wkt, ALAND, AWATER
                FROM mart_socioeconomic_states
                WHERE survey_year = {year}
            """,
            database=DB, workgroup=WG, boto3_session=session
        )
        for metric in METRICS:
            geojson = df_to_geojson(df, metric)
            save_json(geojson, f"{OUTPUT_DIR}/states_{year}_{metric}.json")
        print(f"  states {year} ✓ ({len(df)} rows)")
        del df
        gc.collect()
    except Exception as e:
        print(f"  states {year} FAILED: {e}")

=== Regenerating states data ===
  states 2012 ✓ (52 rows)
  states 2013 ✓ (52 rows)
  states 2014 ✓ (52 rows)
  states 2015 ✓ (52 rows)
  states 2016 ✓ (52 rows)
  states 2017 ✓ (52 rows)
  states 2018 ✓ (52 rows)
  states 2019 ✓ (52 rows)
  states 2020 ✓ (52 rows)
  states 2021 ✓ (52 rows)
  states 2022 ✓ (52 rows)
  states 2023 ✓ (52 rows)
  states 2024 ✓ (52 rows)


Generating county json files

In [9]:
import json
import gc
import time

def query_with_retry(sql, max_retries=3):
    for attempt in range(max_retries):
        try:
            return wr.athena.read_sql_query(
                sql=sql,
                database=DB,
                workgroup=WG,
                boto3_session=session,
            )
        except Exception as e:
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 10
                print(f"    Retry {attempt + 1}/{max_retries} after {wait}s: {str(e)[:80]}")
                time.sleep(wait)
            else:
                raise

print("=== Generating counties data ===")
for year in YEARS:
    try:
        df = query_with_retry(f"""
            SELECT geography_id, geography_name, state_fips, county_fips,
                   survey_year, total_population, median_age,
                   median_household_income, median_home_value,
                   median_gross_rent, pct_white_alone, pct_black_alone,
                   pct_asian_alone, pct_hispanic_or_latino,
                   poverty_rate, bachelors_plus_rate, high_school_rate,
                   owner_occupancy_rate, renter_rate, drove_alone_rate,
                   walked_to_work_rate, remote_work_rate,
                   geometry_wkt, ALAND, AWATER
            FROM mart_socioeconomic_counties
            WHERE survey_year = {year}
        """)
        for metric in METRICS:
            geojson = df_to_geojson(df, metric)
            save_json(geojson, f"{OUTPUT_DIR}/counties_{year}_{metric}.json")
        print(f"  counties {year} ✓ ({len(df)} rows)")
        del df
        gc.collect()
    except Exception as e:
        print(f"  counties {year} FAILED: {e}")

=== Generating counties data ===
  counties 2012 ✓ (3221 rows)
  counties 2013 ✓ (3221 rows)
  counties 2014 ✓ (3220 rows)
  counties 2015 ✓ (3220 rows)
  counties 2016 ✓ (3220 rows)
  counties 2017 ✓ (3220 rows)
  counties 2018 ✓ (3220 rows)
  counties 2019 ✓ (3220 rows)
  counties 2020 ✓ (3221 rows)
  counties 2021 ✓ (3221 rows)
  counties 2022 ✓ (3222 rows)
  counties 2023 ✓ (3222 rows)
  counties 2024 ✓ (3222 rows)


In [10]:
# In notebook - check what properties are in the generated JSON
import json

with open(f"{OUTPUT_DIR}/states_2024_median_household_income.json") as f:
    data = json.load(f)

# Print first feature's properties keys
print(list(data['features'][0]['properties'].keys()))

['geography_id', 'geography_name', 'state_fips', 'survey_year', 'total_population', 'median_age', 'median_household_income', 'median_home_value', 'median_gross_rent', 'pct_white_alone', 'pct_black_alone', 'pct_asian_alone', 'pct_hispanic_or_latino', 'poverty_rate', 'bachelors_plus_rate', 'high_school_rate', 'owner_occupancy_rate', 'renter_rate', 'drove_alone_rate', 'walked_to_work_rate', 'remote_work_rate', 'centroid_lat', 'centroid_lon', 'aland', 'awater', 'fill_color']


In [12]:
import os
size = os.path.getsize("../dashboard/data.duckdb") / (1024*1024*1024)
print(f"DuckDB size: {size:.2f} GB")

DuckDB size: 3.71 GB
